# 07 · One-stage anchor-free detector: Ultralytics YOLO26

Trains YOLO26 on the shared data and scores it with the shared evaluator, so its numbers sit in the same table as Faster R-CNN (02) and RetinaNet (04).

| Section | What it does |
|---|---|
| Setup, protocol | Paths, data config, the YOLO training recipe and budget |
| Experiments | `baseline`, then one change each: input size (960, 640), stride-4 head (`p2_960`), milder scale jitter, then `tuned` |
| Train | `RUN = [...]` cell; also runs unattended with `run_queue.py 07_yolo.ipynb:baseline,imgsz_960,p2_960` |
| Comparison | YOLO runs next to the best Faster R-CNN and RetinaNet runs; per-epoch curves |
| Sanity check | Our evaluator vs Ultralytics' own metric |
| Inference-only checks | NMS vs NMS-free head; larger test size. No training |
| Speed, failure analysis | FPS; error types, worst frames, miss rates, confusions |
| Submission | Test predictions in the competition format |

**How YOLO26 differs from the anchor-based models** (Ultralytics docs): no anchors, so each grid cell predicts box distances directly; no DFL;
two heads trained together, one-to-many (used with NMS, the default) and one-to-one (NMS-free, `nms=False`);
STAL, a label assignment that keeps positives for small targets.

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    SRC_DIR = Path("/kaggle/working/ai-hw2-multiobject-detection/src")
    DATA_ROOT = Path("/kaggle/input/COMPETITION_FOLDER")                    # TODO
    PROC_ROOT = Path("/kaggle/input/PROCESSED_DATASET/data_processed")      # TODO
    RUNS_ROOT = Path("/kaggle/working/runs")
else:
    SRC_DIR = Path.cwd()
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", "../multiple-object-video-detection-hw-2-ucu-ai-2026"))
    PROC_ROOT = Path(os.environ.get("PROC_ROOT", "../data_processed"))
    RUNS_ROOT = Path(os.environ.get("RUNS_ROOT", "../runs"))

sys.path.insert(0, str(SRC_DIR))
from ultralytics import YOLO

from common.analysis import GroundTruth, error_breakdown, error_table, miss_and_confusion, show_frame
from common.data import CLASSES
from common.engine import load_summaries
from common.evaluate import evaluate_detections
from common.submission import write_submission
from common.yolo import measure_fps, predict_coco, predict_test, train_and_score, val_items, write_data_yaml

# FIXTURE=1: a tiny CPU smoke test (few frames, small size, 2 epochs). Never report its numbers.
FIXTURE = os.environ.get("FIXTURE") == "1"
GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS_DIR = RUNS_ROOT / ("yolo_fixture" if FIXTURE else "yolo")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML = write_data_yaml(PROC_ROOT, RUNS_DIR / "data.yaml")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "| fixture:", FIXTURE)

## Protocol

Shared with every model (see `AGENTS.md`): same training frames, original val frames for scoring, the shared evaluator,
`conf=0.001` and 500 detections per image at inference, seed 42, AMP.

YOLO-specific choices:
- **Recipe:** Ultralytics' own defaults for YOLO (optimizer `auto`, its learning rate, mosaic, HSV and scale jitter), just as 02 and 04 use each family's own learning rate.
  Ultralytics accumulates gradients to a nominal batch of 64, so the real batch size (set by `batch`, a fraction of GPU memory) only changes memory use.
- **Augmentation rules from 01:** no vertical flip and no rotation (the camera is tilted). `copy_paste` is off because it needs masks.
- **Budget: 4 epochs for every YOLO experiment.** Measured on our RTX 4060 laptop (8 GB): at 1280 px AutoBatch fits only 2 images per step,
  1.0 s per step, so one epoch takes about 25 min. 4 epochs is ~6,100 steps, the same number of steps as Faster R-CNN's 8 epochs at batch 4,
  and Faster R-CNN's mAP plateaued after epoch 3. YOLO recipes normally train much longer, so the per-epoch curve shows whether it had converged.
- **Short-schedule adjustments:** warm-up shortened from 3 epochs to 0.5 (otherwise 3 of the 4 epochs would run at a reduced learning rate);
  weights updated every 16 images (`nbs=16`, ~760 updates) instead of Ultralytics' default 64 (~190 updates); mosaic off for the last epoch.
  Ultralytics rescales weight decay to the accumulated batch.
- **Which epoch counts:** every epoch is saved (`save_period=1`) and re-scored on the original val frames afterwards; the best epoch by our mAP is the run's result,
  the same rule `engine.py` uses. Ultralytics' own `best.pt` is chosen on the grey-filled val, so we don't use it.

In [ ]:
PROTOCOL = dict(
    epochs=4,                  # same for every YOLO experiment (~25 min/epoch at 1280 on an RTX 4060 laptop)
    batch=0.70,                # fraction of GPU memory (Ultralytics AutoBatch); P2 automatically gets a smaller batch
    train=dict(
        optimizer="auto", seed=42, deterministic=True, amp=True,
        workers=int(os.environ.get("WORKERS", 6)), cache=False,   # 6 on the 32 GB laptop; WORKERS=2 on the 14 GB machine
        fliplr=0.5, flipud=0.0, degrees=0.0, copy_paste=0.0,
        close_mosaic=1,                  # last epoch without mosaic
        warmup_epochs=0.5,               # default 3 would cover most of a 4-epoch run
        nbs=16,                          # optimizer step every 16 images (default 64 -> only ~190 updates in 4 epochs)
        save_period=1, plots=False, val=True, device=DEVICE,
    ),
    predict=dict(conf=0.001, iou=0.7, max_det=500, device=DEVICE, batch=4),
    score_every=1,             # score every saved epoch (set 2 to halve the scoring time)
    val_max_images=None,
)
if FIXTURE:
    PROTOCOL.update(epochs=2, batch=2, val_max_images=6)
    PROTOCOL["train"].update(workers=0, amp=False, close_mosaic=0)
    PROTOCOL["predict"].update(batch=2)

## Experiments

Each experiment changes one thing relative to `baseline`.

| Name | Change | Question it answers |
|---|---|---|
| `baseline` | `yolo26s.pt` (COCO), imgsz 1280 | Same input size as the CNNs |
| `imgsz_960` | imgsz 960 for training and prediction | Required parameter 1: how much does resolution matter for small objects? |
| `imgsz_640` | imgsz 640 | Third point on the resolution curve (640 / 960 / 1280) |
| `p2_960` | `yolo26s-p2.yaml` + COCO weights, imgsz 960; **compare with `imgsz_960`** | Required parameter 2: does a stride-4 head (as in Faster R-CNN's FPN) find more small objects? |
| `p2_1280` | P2 head at 1280 px; **compare with `baseline`** | Same question at full size; needs a 16 GB GPU (RX 6800 XT) |
| `imgsz_1600` | imgsz 1600; **compare with `baseline`** | Does more resolution keep helping? Needs a 16 GB GPU |
| `low_scale_960` | scale jitter 0.25 instead of 0.5, imgsz 960; **compare with `imgsz_960`** | Default jitter zooms out to 0.5×, which shrinks already-tiny objects below the stride; does milder jitter help small objects? |
| `tuned` | fill in from the results | Combination of what helped |

**Why P2 at 960, not 1280:** at 1280 px the 8 GB GPU fits only 2 images per step without P2; with P2 it would drop to about 1, which breaks
batch norm and takes ~3 h. The P2 question only needs the same image size on both sides, so it is answered at 960 against `imgsz_960`.
`p2_1280` and `imgsz_1600` don't fit in 8 GB; they run on the 16 GB RX 6800 XT (same protocol, different GPU, so compare
mAP but not minutes). The inference-only section also tests predicting at 1600 with a model trained at 1280.

**Note on `p2`:** there are no pretrained P2 weights. `.load("yolo26s.pt")` transfers only the layers whose names and shapes match
(the backbone and part of the neck); the rest, including the whole new head, starts from random. So `p2` has a weaker start than `baseline`.

In [ ]:
BASE = dict(model="yolo26s.pt", weights=None, imgsz=1280, train={}, predict={})

EXPERIMENTS = {
    "baseline":   {},
    "imgsz_960":  dict(imgsz=960),
    "imgsz_640":  dict(imgsz=640),
    "p2_960":     dict(model="yolo26s-p2.yaml", weights="yolo26s.pt", imgsz=960),
    "low_scale_960": dict(imgsz=960, train=dict(scale=0.25)),
    "p2_1280":    dict(model="yolo26s-p2.yaml", weights="yolo26s.pt"),     # 16 GB GPU only
    "imgsz_1600": dict(imgsz=1600),                                        # 16 GB GPU only
    "tuned":      {},          # TODO: fill in after the single-change runs
}
if FIXTURE:
    BASE.update(imgsz=320)
    EXPERIMENTS = {"baseline": {}, "p2": dict(model="yolo26s-p2.yaml", weights="yolo26s.pt")}

### Timing (optional, run once)

Trains 10% of one epoch at the baseline settings. Read the speed from the progress bar (`s/it` × steps per epoch); the printed estimate
also multiplies one-off start-up (AutoBatch, label scan, final validation) by 10, so it runs high. Leave it `False` (the queue runner executes this cell too).
Measured on the RTX 4060 laptop: batch 2, 1.0 s/it, ≈ 25 min per epoch at 1280.

In [ ]:
TIME_EPOCH = False
if TIME_EPOCH:
    import time
    t0 = time.time()
    YOLO(BASE["model"]).train(data=str(DATA_YAML), project=str(RUNS_DIR.resolve()), name="_timing", exist_ok=True,
                              imgsz=BASE["imgsz"], epochs=1, batch=PROTOCOL["batch"], fraction=0.1,
                              **{**PROTOCOL["train"], "val": False, "save_period": -1})
    print(f"≈ {(time.time() - t0) / 60 * 10:.1f} min per full epoch (+ ~1 min scoring per epoch)")

## Train

In [ ]:
RUN = ["baseline"]          # then ["imgsz_960", "p2_960", "imgsz_640", "low_scale_960"]; 16 GB GPU: ["p2_1280", "imgsz_1600"]; later ["tuned"]
for name in RUN:
    train_and_score(name, {**BASE, **EXPERIMENTS[name]}, PROTOCOL, DATA_YAML, RUNS_DIR, GT_FILE, DATA_ROOT, YOLO)

## Comparison with the anchor-based models

Best epoch of each YOLO run, next to the best Faster R-CNN and RetinaNet runs (all scored by `common/evaluate.py` on the same 359 original val frames).

In [ ]:
yolo = load_summaries(RUNS_DIR)
others = {fam: load_summaries(RUNS_ROOT / fam) for fam in ["faster_rcnn", "retinanet"]}
rows = {f"yolo/{n}": r["best"] for n, r in yolo.items()}
best_other = {}
for fam, runs in others.items():
    if runs:
        best_other[fam] = max(runs, key=lambda n: runs[n]["best"]["mAP"])
        rows[f"{fam}/{best_other[fam]}"] = runs[best_other[fam]]["best"]

cols = ["epoch", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"] + [f"AP_{c}" for c in CLASSES]
table = pd.DataFrame({n: {c: b.get(c, np.nan) for c in cols} for n, b in rows.items()}).T
table["train_min"] = pd.Series({f"yolo/{n}": r["best"].get("train_minutes_total") for n, r in yolo.items()})
table.style.format({c: "{:.4f}" for c in cols[1:]}, na_rep="").highlight_max(axis=0, subset=cols[1:])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
curves = [("yolo", n, RUNS_DIR / n) for n in yolo] + [(f, n, RUNS_ROOT / f / n) for f, n in best_other.items()]
for fam, name, d in curves:
    if not (d / "history.json").exists():
        continue
    h = pd.DataFrame(json.loads((d / "history.json").read_text()))
    style = "-" if fam == "yolo" else "--"
    for ax, col in zip(axes, ["mAP", "AP_small", "AP_large"]):
        ax.plot(h.epoch, h[col], style, marker="o", ms=3, label=f"{fam}/{name}")
for ax, t in zip(axes, ["val mAP@[.5:.95]", "val AP_small", "val AP_large"]):
    ax.set(xlabel="epoch", ylabel=t, title=t)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observations:** _fill in_

## Sanity check: our evaluator vs Ultralytics' metric

Ultralytics' own mAP is computed on the **grey-filled** val copies, with its own matching and up to 300 detections. It should be in the same range as ours;
a large gap would point to a class-offset or box-format bug in `common/yolo.py`. Report only our numbers.

In [ ]:
check = {}
for n, r in yolo.items():
    b = r["best"]
    check[n] = {"ours mAP": b["mAP"], "ultralytics mAP50-95": b.get("ul_metrics/mAP50-95(B)"),
                "ours AP50": b["AP50"], "ultralytics mAP50": b.get("ul_metrics/mAP50(B)")}
pd.DataFrame(check).T.round(4)

## Inference-only checks (no training)

On the best YOLO run:
- **NMS-free head** (`nms=False`): the one-to-one head, the same idea as DETR's set prediction. It returns at most 300 boxes per image.
- **Larger test size:** predict at 1600 a model trained at 1280 (test-time upscaling, a cheap version of the resolution lever).

In [ ]:
BEST = max(yolo, key=lambda n: yolo[n]["best"]["mAP"]) if yolo else None
print("best YOLO run:", BEST)
if BEST:
    cfg = yolo[BEST]["config"]
    items = val_items(GT_FILE, DATA_ROOT, PROTOCOL["val_max_images"])
    ids = [i for i, _ in items]
    variants = {"as trained": {}, "NMS-free head": {"nms": False},
                "predict at 1600": {"imgsz": 320 if FIXTURE else 1600}}
    checks = {}
    for label, kw in variants.items():
        model = YOLO(str(RUNS_DIR / BEST / "best.pt"))   # fresh object: the NMS mode is fixed at its first predict
        dets = predict_coco(model, items, **{"imgsz": cfg["imgsz"], **PROTOCOL["predict"], **cfg.get("predict", {}), **kw})
        checks[label] = evaluate_detections(GT_FILE, dets, image_ids=ids)
    (RUNS_DIR / BEST / "inference_checks.json").write_text(json.dumps(checks, indent=1))
    display(pd.DataFrame(checks).T[["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]].round(4))

**Observations:** _fill in_

## Speed

Images per second at batch size 1 on 50 val frames preloaded in memory (letterbox, model and NMS included), same frames as 02 and 04.

In [ ]:
items = val_items(GT_FILE, DATA_ROOT)
speed_paths = [p for _, p in items[::max(1, len(items) // 50)][:50]]
fps = {}
for n, r in yolo.items():
    fps[f"yolo/{n}"] = measure_fps(YOLO(str(RUNS_DIR / n / "best.pt")), speed_paths, r["config"]["imgsz"],
                                   device=DEVICE)
pd.Series(fps, name="FPS").round(1)

## Failure analysis

Error types (TIDE-style, `common/analysis.error_breakdown`) at score ≥ 0.3 for every YOLO run, then the worst frames and miss rates of the best run.

In [ ]:
SCORE_THR = 0.3
gt = GroundTruth(GT_FILE)
breakdown = {f"yolo/{n}": error_breakdown(gt, json.loads((RUNS_DIR / n / "val_dets.json").read_text()), SCORE_THR)
             for n in yolo}
for fam, n in best_other.items():
    f = RUNS_ROOT / fam / n / "val_dets.json"
    if f.exists():
        breakdown[f"{fam}/{n}"] = error_breakdown(gt, json.loads(f.read_text()), SCORE_THR)
pd.DataFrame(breakdown).T.round(3)

In [ ]:
ANALYSE = BEST
dets = json.loads((RUNS_DIR / ANALYSE / "val_dets.json").read_text())
errors = error_table(gt, dets, SCORE_THR)
print("Totals over val:", errors[["n_gt", "missed", "fp", "wrong_class"]].sum().to_dict())
errors.groupby("video")[["n_gt", "missed", "fp", "wrong_class"]].sum()

In [ ]:
for col, title in [("missed", "Most missed objects"), ("fp", "Most false positives"), ("wrong_class", "Most class confusions")]:
    worst = errors.sort_values(col, ascending=False).iloc[0]
    show_frame(gt, dets, int(worst.image_id), DATA_ROOT, SCORE_THR, f"{title}: {int(worst[col])} of {int(worst.n_gt)} objects")

In [ ]:
miss, confusion = miss_and_confusion(gt, dets, SCORE_THR)
print("Miss rate by class and size:"); display(miss.round(3))
if len(confusion):
    print("Class confusions (true -> predicted):"); display(confusion)

**Observations:** _fill in_

## Test submission

Writes `runs/yolo/<SUBMIT>/submission.csv` (classes 0–7, original pixels, ≤ 500 boxes per image).

In [ ]:
SUBMIT = BEST
cfg = yolo[SUBMIT]["config"]
preds = predict_test(YOLO(str(RUNS_DIR / SUBMIT / "best.pt")), TEST_CSV, DATA_ROOT,
                     max_images=2 if FIXTURE else None,
                     **{"imgsz": cfg["imgsz"], **PROTOCOL["predict"], **cfg.get("predict", {})})
sub_path = RUNS_DIR / SUBMIT / "submission.csv"
sub = write_submission(preds, TEST_CSV, sub_path)

n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) // 6
classes = {int(t) for s in sub.PredictionString.dropna() for t in s.split()[0::6]}
assert classes <= set(range(8)), classes
assert n_boxes.max() <= 500
print(f"Wrote {sub_path}: {len(sub)} rows, {int((n_boxes > 0).sum())} with detections, "
      f"max {int(n_boxes.max())} boxes per image, classes {sorted(classes)}")